# Concept fractures: circuit tracing on a ROME-edited Qwen3-0.6B

Prototyping notebook. Finds which components carry the fact before and after a single ROME edit,
using Latium's subject-noise corruption (as in causal tracing) and the metric

`LD = logit(target_new) - logit(target_true)` at the last token.

- **Activation patching**: in the corrupted run, restore one clean attention / MLP output at one
  (layer, token) and measure how much of the clean-vs-corrupt LD gap comes back (1 = fully restored).
- **Head patching**: the same, per attention head (restored at all positions), plus the
  first-order **attribution patching** estimate for comparison. The gradient estimate tracks real patching
  for mid/late heads but breaks down in the first layers, next to the heavily noised embeddings, so the
  real map is the one to read.

Both are run on the unedited model (circuit for " Paris") and the edited model (circuit for " Rome"),
with the same noise draws, so the maps are paired.

In [ ]:
from pathlib import Path
import sys

import os
os.environ["HIP_VISIBLE_DEVICES"] = "0"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from omegaconf import OmegaConf

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.causal_trace.model_adapter import (
    corrupt_hook,
    embedding_std,
    hidden_from_output,
    make_noise_samples,
    module_dict,
    prepare_inputs,
    repeat_inputs,
    replace_hidden,
    temporary_hooks,
)
from src.causal_trace.tokenization import find_subject_span, target_token_ids
from src.common.model_config import load_model_config
from src.editing.rome import RomeEditMethod
from src.handlers.rome import ModelHandler
from src.runtime import set_global_seed

## Parameters

In [ ]:
MODEL_KEY = "qwen3-0.6b"
SEED = 42
MODELS_DIR = ROOT.parent / "models"  # local HF cache used by the rest of Latium
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = None  # e.g. 20 for a quick smoke test; None keeps the model config value

# (prompt_template, subject, " target_new", " target_true")
FACT = ("{} is in", "The Eiffel Tower", " Rome", " Paris")
N_NOISE = 10  # paired noise draws per corrupted run
# Noise std = NOISE_MULT x embedding std (3.0 as in Meng et al., 2022). The calibrated
# cfg.model.corruption_noise_multiplier (~0.08) is too weak here: LD barely moves.
NOISE_MULT = 3.0
TOP_HEADS = 10

## Load model via Latium

In [ ]:
set_global_seed(SEED)
CONFIG_DIR = ROOT / "src" / "config"
cfg = OmegaConf.create(
    {
        "seed": SEED,
        "runtime": {},
        "model": load_model_config(MODEL_KEY),
        "generation": OmegaConf.load(CONFIG_DIR / "generation" / "generation.yaml"),
    }
)
cfg.model.models_dir = str(MODELS_DIR)
cfg.model.device = DEVICE
if EPOCHS is not None:
    cfg.model.epochs = EPOCHS

handler = ModelHandler(cfg)
model, tok = handler.model, handler.tokenizer
EDIT_LAYER = int(handler._layer)
print(cfg.model.name, "| edit layer:", EDIT_LAYER, "| device:", handler.device)

## Prompt, corruption and metric

In [ ]:
modules = module_dict(model)
EMBED = modules[cfg.model.corrupt_layer_name_template]
LAYERS = model.model.layers
N_LAYERS = model.config.num_hidden_layers
N_HEADS = model.config.num_attention_heads
HEAD_DIM = model.config.head_dim

PROMPT = FACT[0].format(FACT[1])
inputs = prepare_inputs(handler, PROMPT)
batch = repeat_inputs(inputs, N_NOISE)
SEQ = inputs["input_ids"].shape[1]
TOKENS = [tok.decode(t) for t in inputs["input_ids"][0]]
span = find_subject_span(tok, PROMPT, FACT[1])
T_NEW = target_token_ids(tok, FACT[2], prompt=PROMPT)[0]
T_TRUE = target_token_ids(tok, FACT[3], prompt=PROMPT)[0]

noise_std = embedding_std(handler, modules) * NOISE_MULT
noise = make_noise_samples(
    num_samples=N_NOISE,
    subject_length=len(span.positions),
    hidden_size=model.config.hidden_size,
    noise_std=noise_std,
    device=handler.device,
    dtype=model.dtype,
    seed=SEED,
)
print(TOKENS, "| subject positions:", span.positions, "| targets:", tok.decode(T_NEW), tok.decode(T_TRUE))


def logit_diff(logits: torch.Tensor) -> torch.Tensor:
    last = logits[:, -1].float()
    return (last[:, T_NEW] - last[:, T_TRUE]).mean()


def run(corrupt: bool = False, hooks=()) -> float:
    if corrupt:
        hooks = [(EMBED, corrupt_hook(span.positions, noise)), *hooks]
    with torch.no_grad(), temporary_hooks(hooks):
        return logit_diff(model(**batch, use_cache=False).logits).item()

## Activation patching (attention / MLP outputs per layer x token)

In [ ]:
COMPONENTS = {"attn": lambda l: LAYERS[l].self_attn, "mlp": lambda l: LAYERS[l].mlp}


def capture_clean() -> dict:
    cache = {}

    def saver(key):
        def hook(_module, _input, output):
            cache[key] = hidden_from_output(output)[:1].detach().clone()

        return hook

    hooks = [(COMPONENTS[k](l), saver((k, l))) for k in COMPONENTS for l in range(N_LAYERS)]
    with torch.no_grad(), temporary_hooks(hooks):
        model(**inputs, use_cache=False)
    return cache


def patch_hook(pos: int, clean: torch.Tensor):
    def hook(_module, _input, output):
        h = hidden_from_output(output).clone()
        h[:, pos] = clean[:, pos]
        return replace_hidden(output, h)

    return hook


def activation_patching() -> dict:
    cache = capture_clean()
    res = {"clean": run(), "corrupt": run(corrupt=True)}
    gap = res["clean"] - res["corrupt"]
    for k in COMPONENTS:
        res[k] = np.zeros((N_LAYERS, SEQ))
        for l in range(N_LAYERS):
            for p in range(SEQ):
                ld = run(corrupt=True, hooks=[(COMPONENTS[k](l), patch_hook(p, cache[(k, l)]))])
                res[k][l, p] = (ld - res["corrupt"]) / gap
    return res


def plot_patching(maps: dict, title: str, diverging: bool = False):
    fig, axes = plt.subplots(1, len(COMPONENTS), figsize=(5 * len(COMPONENTS), 6), sharey=True)
    for ax, k in zip(axes, COMPONENTS):
        m = maps[k]
        lim = np.abs(m).max() if diverging else None
        im = ax.imshow(m, aspect="auto", origin="lower", cmap="RdBu_r" if diverging else "viridis",
                       vmin=-lim if diverging else None, vmax=lim)
        ax.axhline(EDIT_LAYER, color="w" if not diverging else "k", ls=":", lw=1)
        ax.set_xticks(range(SEQ), [repr(t) for t in TOKENS], rotation=45, ha="right")
        ax.set_title(k)
        fig.colorbar(im, ax=ax)
    axes[0].set_ylabel("layer")
    fig.suptitle(title)
    plt.tight_layout()

## Head patching and attribution patching

In [ ]:
def o_proj_inputs(corrupt: bool, grad: bool) -> tuple[dict, torch.Tensor]:
    """Run once, returning each layer's o_proj input (per-head attention outputs, concatenated) and LD."""
    acts, handles = {}, []

    def grab(l):
        def pre_hook(_module, args):
            acts[l] = args[0]
            if grad:
                args[0].retain_grad()

        return pre_hook

    for l in range(N_LAYERS):
        handles.append(LAYERS[l].self_attn.o_proj.register_forward_pre_hook(grab(l)))
    if corrupt:
        handles.append(EMBED.register_forward_hook(corrupt_hook(span.positions, noise)))
    if grad:
        # weights are frozen; make the embeddings the leaf so gradients reach every head
        handles.append(EMBED.register_forward_hook(lambda _m, _i, out: out.detach().requires_grad_()))
    try:
        with torch.set_grad_enabled(grad):
            ld = logit_diff(model(**batch, use_cache=False).logits)
            if grad:
                ld.backward()
    finally:
        for h in handles:
            h.remove()
    return acts, ld.detach()


def head_attribution() -> np.ndarray:
    """Estimated normalised effect of restoring each head (all positions): (clean - corrupt) . dLD/da."""
    clean, ld_clean = o_proj_inputs(corrupt=False, grad=False)
    corrupt, ld_corrupt = o_proj_inputs(corrupt=True, grad=True)
    gap = (ld_clean - ld_corrupt).item()
    attr = np.zeros((N_LAYERS, N_HEADS))
    for l in range(N_LAYERS):
        a = corrupt[l]
        contrib = ((clean[l] - a.detach()) * a.grad).float()
        # LD already averages over noise draws, so the gradient carries the 1/N_NOISE: sum, don't mean
        attr[l] = contrib.view(N_NOISE, SEQ, N_HEADS, HEAD_DIM).sum((0, 1, 3)).cpu().numpy() / gap
    return attr


def patch_head(l: int, h: int, clean: dict, ld_clean: float, ld_corrupt: float) -> float:
    """Real (normalised) effect of restoring head h of layer l at all positions in the corrupted run."""
    sl = slice(h * HEAD_DIM, (h + 1) * HEAD_DIM)

    def pre_hook(_module, args):
        x = args[0].clone()
        x[..., sl] = clean[l][..., sl]
        return (x,) + args[1:]

    handle = LAYERS[l].self_attn.o_proj.register_forward_pre_hook(pre_hook)
    try:
        ld = run(corrupt=True)
    finally:
        handle.remove()
    return (ld - ld_corrupt) / (ld_clean - ld_corrupt)


def head_patching() -> np.ndarray:
    """Real normalised effect of restoring each head (all positions), [n_layers, n_heads]."""
    clean, _ = o_proj_inputs(corrupt=False, grad=False)
    ld_clean, ld_corrupt = run(), run(corrupt=True)
    return np.array([[patch_head(l, h, clean, ld_clean, ld_corrupt) for h in range(N_HEADS)] for l in range(N_LAYERS)])


def top_heads(patched: np.ndarray, attr: np.ndarray, k: int = TOP_HEADS) -> pd.DataFrame:
    idx = np.argsort(-np.abs(patched), axis=None)[:k]
    rows = [dict(zip(("layer", "head"), divmod(int(i), N_HEADS))) for i in idx]
    for r in rows:
        r.update(patched=patched[r["layer"], r["head"]], attribution=attr[r["layer"], r["head"]])
    return pd.DataFrame(rows)


def plot_attribution_check(patched: np.ndarray, attr: np.ndarray, title: str):
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(patched.ravel(), attr.ravel(), s=8, alpha=0.6)
    r = np.corrcoef(patched.ravel(), attr.ravel())[0, 1]
    ax.set_xlabel("head patching")
    ax.set_ylabel("attribution patching")
    fig.suptitle(f"{title} (r = {r:.2f})")
    plt.tight_layout()


def plot_heads(maps: dict):
    fig, axes = plt.subplots(1, len(maps), figsize=(5 * len(maps), 6), sharey=True)
    for ax, (name, m) in zip(axes, maps.items()):
        lim = np.abs(m).max()
        im = ax.imshow(m, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
        ax.axhline(EDIT_LAYER, color="k", ls=":", lw=1)
        ax.set_xlabel("head")
        ax.set_title(name)
        fig.colorbar(im, ax=ax)
    axes[0].set_ylabel("layer")
    plt.tight_layout()

## Baseline (unedited) circuit

In [ ]:
patch_before = activation_patching()
heads_before, attr_before = head_patching(), head_attribution()
print(f"LD clean {patch_before['clean']:.2f} | corrupt {patch_before['corrupt']:.2f}")
plot_patching(patch_before, "before edit: fraction of LD gap restored")
plot_attribution_check(heads_before, attr_before, "before edit")
display(top_heads(heads_before, attr_before))

## Apply ROME edit

In [ ]:
case = {
    "fact_tuple": FACT,
    "target_new_str": FACT[2].strip(),
    "target_true_str": FACT[3].strip(),
    "paraphrase_prompts": [],
    "neighborhood_prompts": [],
}
method = RomeEditMethod()
outcome = method.apply(handler, case)  # edits weights in place
metrics = method.evaluate(handler, case, outcome)
print(outcome.metadata)
metrics

## Edited circuit

In [ ]:
patch_after = activation_patching()
heads_after, attr_after = head_patching(), head_attribution()
print(f"LD clean {patch_after['clean']:.2f} | corrupt {patch_after['corrupt']:.2f}")
plot_patching(patch_after, "after edit: fraction of LD gap restored")
plot_attribution_check(heads_after, attr_after, "after edit")
display(top_heads(heads_after, attr_after))

## Before vs. after

Maps are normalised by their own clean-vs-corrupt gap (before: preference for " Paris",
after: preference for " Rome"), so the difference shows how the *share* of the effect moved between components.

In [ ]:
plot_patching({k: patch_after[k] - patch_before[k] for k in COMPONENTS}, "after - before", diverging=True)
plot_heads({"before": heads_before, "after": heads_after, "after - before": heads_after - heads_before})

## Restore original weights

In [ ]:
for name, W in outcome.restorations.items():
    handler._get_module(name).weight = torch.nn.Parameter(W)
handler.remove_hooks()

assert abs(run() - patch_before["clean"]) < 1e-2, "restore mismatch"
print("restored OK")

## Next steps
- Path patching between the top heads and the edited MLP to separate direct from mediated effects.
- Per-position head patching (subject vs. last token).
- Paraphrase / neighbourhood prompts, and a sweep over facts sharing a subject.